In [1]:
import importlib.util


packages = [
    "cloudpickle", "joblib", "keras", "matplotlib","numpy", "pandas", "pillow", "plotly", "scikit-learn", 
   "tensorflow", "torch", ]

print(f"{'PACKAGE':<20} {'STATUS':<10} {'VERSION'}")
print("-" * 40)

for pkg in packages:
    # handle package names that differ from import names
    import_name = pkg
    if pkg == "beautifulsoup4": import_name = "bs4"
    if pkg == "imbalanced-learn": import_name = "imblearn"
    if pkg == "opencv-python": import_name = "cv2"
    if pkg == "pillow": import_name = "PIL"
    if pkg == "scikit-image": import_name = "skimage"
    if pkg == "scikit-learn": import_name = "sklearn"

    spec = importlib.util.find_spec(import_name)
    if spec is None:
        print(f"{pkg:<20} MISSING")
    else:
        try:
            module = importlib.import_module(import_name)
            version = getattr(module, "__version__", "Unknown")
            print(f"{pkg:<20} Found   {version}")
        except ImportError:
            print(f"{pkg:<20} Error")

PACKAGE              STATUS     VERSION
----------------------------------------
cloudpickle          Found   3.1.2
joblib               Found   1.5.3
keras                Found   3.13.2
matplotlib           Found   3.10.0
numpy                Found   2.0.2
pandas               Found   2.2.2
pillow               Found   11.3.0
plotly               Found   5.24.1
scikit-learn         Found   1.6.1
tensorflow           Found   2.19.0
torch                Found   2.11.0+cpu


In [1]:
import os
import requests
import cloudpickle
import pandas as pd
import numpy as np

print(" Select the environment to load the model from:")
print("1: Local Environment")
print("2: GitHub (Downloads directly from repository)")
env_choice = input("Enter 1 or 2: ").strip()

if env_choice == '1':
    model_path = input(r"Enter local model path (leave blank for default D:\SDA3\2026 Apl\EEWS_Himalayas.pkl): ").strip()
    if not model_path:
        model_path = r'D:\SDA3\2026 Apl\EEWS_Himalayas.pkl'
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found at: {model_path}")
    
    print(f"\n Loading model locally from: {model_path}")
    with open(model_path, 'rb') as f:
        model = cloudpickle.load(f)

elif env_choice == '2':
    # Using the raw.githubusercontent.com URL to get the actual .pkl file, not the HTML page
    github_url = 'https://raw.githubusercontent.com/PavanMohanN/EEW_system_Variational/e31328834ac42e8eecb856f357fe9b253d564fbe/EEWS_Himalayas.pkl'
    print("\n Downloading model from GitHub... This might take a moment.")
    response = requests.get(github_url)
    
    if response.status_code == 200:
        model = cloudpickle.loads(response.content)
        print(" Model successfully loaded from GitHub.")
    else:
        raise Exception(f" Failed to download model. HTTP Status Code: {response.status_code}")
else:
    raise ValueError("Invalid selection. Please run the cell again and enter 1 or 2.")
    
print("Model loaded successfully. Type:", type(model))

 Select the environment to load the model from:
1: Local Environment
2: GitHub (Downloads directly from repository)

 Model successfully loaded from GitHub.
Model loaded successfully. Type: <class '__main__.EEWS'>


In [2]:
print("⚙️ Select Data Input Method:")
print("1: Load from CSV (e.g., Inputs_MappingLayer(P-wave @ 8).csv)")
print("2: Use Default Single Sample")
input_choice = input("Enter 1 or 2: ").strip()

if input_choice == '1':
    csv_filename = input("Enter CSV filename (leave blank for default Inputs_MappingLayer(P-wave @ 8).csv): ").strip()
    if not csv_filename:
        csv_filename = 'Inputs_MappingLayer(P-wave @ 8).csv'
        
    if os.path.exists(csv_filename):
        print(f"\n Loading input data from {csv_filename}")
        X_new = pd.read_csv(csv_filename)
    else:
        raise FileNotFoundError(f"❌ CSV file not found: {csv_filename}. Please ensure it is in the same directory.")
else:
    print("\n Using default single sample input.")
    new_sample = {
        'PGA': 0.106792,
        'PGD': 0.926489,
        'CAV': 32.869158,
        'Ia': 2.962098,
        'Fp': 3.125,
        'Tsig': 1.387797,
        'Sc': 0,
        'dir': 0
    }
    X_new = pd.DataFrame([new_sample])

# Ensure categorical columns are strictly strings
for c in ['Sc', 'dir']:
    if c in X_new.columns:
        X_new[c] = X_new[c].astype(str)

print("\nInput data ready for prediction (first 5 rows):")
display(X_new.head())

⚙️ Select Data Input Method:
1: Load from CSV (e.g., Inputs_MappingLayer(P-wave @ 8).csv)
2: Use Default Single Sample

 Using default single sample input.

Input data ready for prediction (first 5 rows):


,PGA,PGD,CAV,Ia,Fp,Tsig,Sc,dir
0,0.106792,0.926489,32.869158,2.962098,3.125,1.387797,0,0


In [3]:
# UPDATE THESE TUPLES with the actual (min, max) values from your training dataset
training_bounds = {
    'PGA': (0.001, 3.0),
    'PGD': (0.001, 100.0),
    'CAV': (0.1, 150.0),
    'Ia': (0.01, 50.0),
    'Fp': (0.1, 25.0),
    'Tsig': (0.01, 15.0)
}

print("Checking data bounds against training limits...")
out_of_bounds_flag = False

for col in training_bounds:
    if col in X_new.columns:
        min_val, max_val = training_bounds[col]
        
        # Identify rows where the feature is out of bounds
        invalid_rows = X_new[(X_new[col] < min_val) | (X_new[col] > max_val)]
        
        if not invalid_rows.empty:
            out_of_bounds_flag = True
            print(f"  WARNING: Found {len(invalid_rows)} row(s) where '{col}' is outside the reliable training range ({min_val} to {max_val}).")

if not out_of_bounds_flag:
    print("  All input features are within the reliable training bounds.")
    
print("-" * 60)

Checking data bounds against training limits...
  All input features are within the reliable training bounds.
------------------------------------------------------------


In [4]:
y_pred = model.predict(X_new)

# Formatting the output column names
if isinstance(y_pred, np.ndarray):
    if hasattr(model, 'target_names'):
        cols = model.target_names
    else:
        cols = [f"t{i}" for i in range(y_pred.shape[1])]
    y_pred = pd.DataFrame(y_pred, columns=cols)

print("\n Predictions Generated Successfully!")
display(y_pred.head())


 Predictions Generated Successfully!


,0.010,0.015,0.020,0.030,0.040,0.050,0.060,0.075,0.090,0.100,...,0.750,0.800,0.900,1.000,1.200,1.500,2.000,2.500,3.000,4.000
0,0.247402,0.247177,0.259432,0.259558,0.278475,0.297235,0.353753,0.356609,0.468473,0.396168,...,0.278956,0.343371,0.344053,0.235834,0.138434,0.081211,0.079471,0.051926,0.039057,0.021651
